# <u> "_Stars Appearing_": your sky, sonified </u>

The "_Stars Appearing_" piece from the "_Audible Universe_" planetarium show, for
**any site and any night**, with an animation to match: every star above your
horizon sounds a note as it appears, brightest first, and lights up where it
really is in the sky.

**Set the form below, then `Runtime` &rarr; `Run all`.** A preview-sized render
takes a couple of minutes; the first run also downloads the star catalogue and
sky map (~100 MB), which are then kept.

> Curious how it works? `StarsAppearingLocal.ipynb` beside this one is the same
> run with every step in the open.

In [ ]:
import sys

# Where the pieces come from. These two are the only thing to change if this
# notebook moves to a different repository or branch.
STRAUSS_REF = ("git+https://github.com/james-trayford/strauss.git"
               "@local_stars_appearing_animation")
HELPER_URL = ("https://raw.githubusercontent.com/Audio-Universe/"
              "sonified-night-sky/main/StarsAppearingLocal.py")

if "google.colab" in sys.modules:
    from pathlib import Path

    # both steps are skipped once they are done, so a second `Run all` after
    # changing a setting does not pay for the install again
    try:
        import skyfield, strauss
    except ImportError:
        !pip install --quiet "strauss @ {STRAUSS_REF}" skyfield==1.53

    if not Path("StarsAppearingLocal.py").exists():
        !wget --quiet -O StarsAppearingLocal.py "{HELPER_URL}"

In [ ]:
#@title Settings: where, when, and what to render { display-mode: "form" }
#@markdown ### Observer Location and Time
latitude = 53.1143737  #@param {type:"number"}
longitude = -1.2219389  #@param {type:"number"}
#@markdown Latitude is +ve north, longitude +ve *east* - 1.22&deg; west is `-1.22`.
date_time = "2026-09-19 19:00:00"  #@param {type:"string"}
time_zone = "Europe/London"  #@param {type:"string"}
#@markdown `facing` is the centre of the panorama, and of the stereo image.
facing = "S"  #@param ["N","NNE","NE","ENE","E","ESE","SE","SSE","S","SSW","SW","WSW","W","WNW","NW","NNW"]
#@markdown Higher `mag_limit` includes more, dimmer stars.
mag_limit = 4  #@param {type:"slider", min:1, max:7, step:0.5}

#@markdown ### Sonification Properties
duration = 45  #@param {type:"number"}
system = "stereo"  #@param ["mono","stereo","5.1","7.1"]
#@markdown `sound` is the instrument and chord. `Night Harp` takes both from
#@markdown the Sonification Suite's style of that name; `Stars Appearing` is
#@markdown the glockenspiel of the original planetarium piece.
sound = "Night Harp"  #@param ["Night Harp", "Stars Appearing"]

#@markdown ### The picture
#@markdown `size` is `fast_preview` 512&times;256, `preview` 1024&times;512, or
#@markdown `full` - the star map's own 4096&times;2048.
size = "preview"  #@param ["fast_preview","preview","full"]
fps = 15  #@param {type:"integer"}
#@markdown `output` picks the equirectangular `panorama`, the fisheye `dome`
#@markdown master, or `both` from a single pass of the frame generator.
output = "both"  #@param ["panorama","dome","both"]
#@markdown `sky_exposure` is how bright the generated sky comes out, and
#@markdown `horizon` blacks out everything below the horizon.
sky_exposure = 0.75  #@param {type:"number"}
horizon = False  #@param {type:"boolean"}

outdir = "stars_appearing_preview"  #@param {type:"string"}

from StarsAppearingLocal import Config, make_sequence, show_videos, facing_degrees, CARDINALS

cfg = Config(latitude=latitude, longitude=longitude, date_time=date_time,
             time_zone=time_zone, facing=facing, mag_limit=mag_limit,
             duration=duration, system=system, size=size, fps=fps,
             output=output, sky_exposure=sky_exposure, horizon=horizon,
             outdir=outdir)

### <u> Making it </u>

The sky, the sonification and the animation, in one go. The bars below are the
star map coming down, the notes rendering, and the frames going out to `ffmpeg`.

In [ ]:
result = make_sequence(cfg, sound=sound)

### <u> The sky you asked for </u>

Every star that sounded, laid out as it appears on the panorama &mdash; the
facing direction in the middle, the horizon along the bottom &mdash; and then the
sky the animation is drawn over. If these are not the sky you meant, change the
form above and run again.

In [ ]:
import matplotlib.pyplot as plt

sky = result.sky

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_facecolor("#0b0c15")

x = (sky["az"] - facing_degrees(cfg.facing) - 180) % 360

ax.scatter(x, sky["alt"], s=40 * 10 ** (-0.2 * sky["magnitude"]),
           c=sky["bv"], cmap="RdYlBu_r", vmin=-1.5, vmax=2.5, lw=0)
ax.set_xticks([(360 * i / 16 - facing_degrees(cfg.facing) - 180) % 360
               for i in range(16)])
ax.set_xticklabels(CARDINALS, fontsize=8)
ax.set_xlim(0, 360)
ax.set_ylim(0, 90)
ax.set_xlabel("compass direction")
ax.set_ylabel("altitude [deg]")
ax.set_title(f"{len(sky)} stars over {cfg.latitude:.2f}, {cfg.longitude:.2f} "
             f"at {cfg.date_time}")
plt.show()

plt.figure(figsize=(14, 14 * cfg.height / cfg.width))
plt.imshow(plt.imread(result.background))
plt.axis("off")
plt.title(f"the sky over {cfg.latitude:.2f}, {cfg.longitude:.2f} "
          f"at {cfg.date_time}, facing {cfg.facing}")
plt.show()

### <u> The finished sequence </u>

In [ ]:
show_videos(result.videos)

### <u> Keeping it </u>

Everything is written to the `outdir` you set above:

- `stars_appearing.wav` &mdash; the sonification on its own
- `stars_appearing_panorama.mp4` &mdash; equirectangular, 360&deg; &times; 180&deg;
- `stars_appearing_dome.mp4` &mdash; the fisheye planetarium master

On _Colab_ these live on a temporary machine, so download anything you want to
keep from the file browser on the left before the session ends.

Render at `preview` while you are still choosing a site and a night, then set
`size` to `full` and raise `fps` for the final pass &mdash; that one takes
considerably longer.